# VectorStore Retriever — 벡터 기반 문서 검색기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **VectorStoreRetriever**를 `as_retriever()`로 생성하는 방법 이해하기
- Similarity, MMR, Score Threshold 세 가지 검색 방식의 차이 파악하기
- `ConfigurableField`로 런타임에 검색 파라미터를 동적으로 바꾸기

## 사전 지식

- 6-4 VectorStore(벡터 저장소)에서 FAISS 벡터스토어를 만들어본 경험
- 임베딩(Embedding)이 텍스트를 숫자 벡터로 변환한다는 개념

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> E[쿼리 임베딩]:::process
    E --> V[(FAISS<br/>벡터스토어)]:::storage
    V --> S{검색 방식}:::process
    S -- Similarity --> R1[유사도<br/>Top-K]:::output
    S -- MMR --> R2[다양성<br/>고려]:::output
    S -- Threshold --> R3[임계값<br/>필터링]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

`VectorStoreRetriever`는 벡터 데이터베이스를 활용하여 의미적으로 유사한 문서를 검색하는 가장 기본적인 Retriever(검색기)입니다.

## 주요 특징

| 특징 | 설명 |
|------|------|
| 의미 기반 검색 | 임베딩 벡터 간 유사도(코사인 유사도)로 검색 |
| 다양한 검색 방식 | Similarity, MMR, Score Threshold 지원 |
| 동적 설정 | `ConfigurableField`로 런타임 파라미터 조정 |
| 통합 인터페이스 | 모든 벡터 DB에서 동일한 API 사용 |

## 환경 설정


In [1]:
import os
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")


## 1. VectorStore 생성

먼저 문서를 로드하고 벡터 데이터베이스를 생성합니다.


In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# ---------------------------------------------------
# 문서를 로드하여 VectorStore 생성을 위한 원본 텍스트 준비
# ---------------------------------------------------

# ============================================================
# TODO: 문서를 로드하고 분할 수와 전체 길이를 출력하세요
# 힌트: TextLoader로 파일을 읽은 뒤 .load() 호출
# 예상 결과: 로드된 문서 수와 문자 수 출력
# ============================================================

# 1단계: 문서 로드
# TextLoader: 텍스트 파일을 Document 객체로 변환하는 로더
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

print(f"로드된 문서 수: {len(documents)}")
print(f"문서 전체 길이: {len(documents[0].page_content)} 문자")

로드된 문서 수: 1
문서 전체 길이: 7482 문자


In [3]:
# ---------------------------------------------------
# 문서를 작은 청크로 분할하여 검색 단위 생성
# ---------------------------------------------------

# ============================================================
# TODO: RecursiveCharacterTextSplitter로 문서를 분할하세요
# 힌트: chunk_size=500, chunk_overlap=50 설정
# 예상 결과: 분할된 청크 수와 첫 번째 청크 내용 출력
# ============================================================

# 2단계: 문서 분할
# RecursiveCharacterTextSplitter: 단락, 줄바꿈, 공백 순서로 재귀 분할
# chunk_overlap: 인접 청크 간 중복 문자 수 (문맥 연결 유지)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

split_docs = text_splitter.split_documents(documents)

print(f"\n분할된 청크 수: {len(split_docs)}")
print(f"\n첫 번째 청크 내용:")
print("=" * 60)
print(split_docs[0].page_content)
print("=" * 60)


분할된 청크 수: 19

첫 번째 청크 내용:
Scikit Learn

Scikit-learn은 Python 언어를 위한 또 다른 핵심 라이브러리로, 기계 학습의 다양한 알고리즘을 구현하기 위해 설계되었습니다. 이 라이브러리는 2007년 David Cournapeau에 의해 프로젝트가 시작되었으며, 그 후로 커뮤니티의 광범위한 기여를 받아 현재까지 발전해왔습니다. Scikit-learn은 분류, 회귀, 군집화, 차원 축소 등 다양한 기계 학습 작업을 지원하며, 사용하기 쉬운 API와 함께 제공되어 연구자와 개발자가 복잡한 데이터 과학 문제를 해결할 수 있도록 돕습니다.


In [4]:
# 3단계: OpenAI 임베딩 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4단계: FAISS 벡터 데이터베이스 생성
vectorstore = FAISS.from_documents(split_docs, embeddings)

print("\n✅ 벡터 데이터베이스 생성 완료")
print(f"저장된 벡터 수: {vectorstore.index.ntotal}")



✅ 벡터 데이터베이스 생성 완료
저장된 벡터 수: 19


## 2. VectorStoreRetriever 생성

`as_retriever()` 메서드는 VectorStore를 Retriever로 변환합니다.

### 주요 파라미터

- `search_type`: 검색 방식 선택
  - `"similarity"`: 유사도 기반 검색 (기본값)
  - `"mmr"`: Maximal Marginal Relevance - 다양성 고려
  - `"similarity_score_threshold"`: 임계값 이상의 문서만 반환

- `search_kwargs`: 검색 옵션 설정
  - `k`: 반환할 문서 수 (기본값: 4)
  - `score_threshold`: 최소 유사도 점수 (0~1)
  - `fetch_k`: MMR의 후보 문서 수 (기본값: 20)
  - `lambda_mult`: MMR 다양성 파라미터 (0~1, 기본값: 0.5)

> 🎯 **강의 포인트**: `as_retriever()`는 VectorStore를 LangChain의 표준 Retriever 인터페이스로 변환합니다. 어떤 벡터스토어(FAISS, Chroma, Pinecone 등)를 사용해도 동일한 API로 검색할 수 있어요.

> 🔑 **핵심 개념**: `search_type`과 `search_kwargs`의 조합으로 검색 동작을 완전히 제어합니다. 기본값(similarity, k=4)으로 시작해서 상황에 맞게 조정하세요.

In [5]:
# ---------------------------------------------------
# VectorStore를 Retriever 인터페이스로 변환
# ---------------------------------------------------

# ============================================================
# TODO: as_retriever()로 기본 retriever를 생성하고 설정을 확인하세요
# 힌트: vectorstore.as_retriever() — 기본값은 similarity, k=4
# 예상 결과: search_type과 search_kwargs 출력
# ============================================================

# as_retriever(): VectorStore → Retriever 변환
# search_type 기본값: "similarity" (코사인 유사도)
# search_kwargs 기본값: {"k": 4}
retriever = vectorstore.as_retriever()

print("✅ Retriever 생성 완료")
print(f"검색 타입: {retriever.search_type}")
print(f"검색 설정: {retriever.search_kwargs}")

✅ Retriever 생성 완료
검색 타입: similarity
검색 설정: {}


## 3. 기본 검색 - Similarity Search

가장 유사한 벡터를 찾아 관련 문서를 반환합니다.


In [6]:
# 쿼리: Transformer 모델에 대한 질문
query = "Transformer 모델은 어떤 메커니즘을 기반으로 하나요?"

# invoke() 메서드로 문서 검색
docs = retriever.invoke(query)

print(f"📝 검색 쿼리: {query}")
print(f"\n검색된 문서 수: {len(docs)}")
print("\n" + "=" * 80)

for i, doc in enumerate(docs, 1):
    print(f"\n[문서 {i}]")
    print(doc.page_content[:200] + "...")
    print("-" * 80)


📝 검색 쿼리: Transformer 모델은 어떤 메커니즘을 기반으로 하나요?

검색된 문서 수: 4


[문서 1]
"Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다. 또한, 논문은 전통적인 RNN이나 CNN에 의존하지 않음으...
--------------------------------------------------------------------------------

[문서 2]
어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Attention)'은 입력 데이터 내의 각 요소가 서로 얼마나 관련이 있는지를 계산하여, 모델이 문맥을 더 잘 이해할 수 있게 합니다. 이는 특히 문장 내에서 단어 간의 관계를 파악하는...
--------------------------------------------------------------------------------

[문서 3]
Attention is all you need

"Attention Is All You Need"는 2017년에 발표된 논문으로, 변환기(Transformer) 모델의 아키텍처를 처음으로 소개한 연구입니다. Ashish Vaswani와 그의 공동 연구자들에 의해 작성된 이 논문은 자연어 처리(NLP)와 기계 학습 분야에 큰 영향을 미쳤습니다. 변환기 모델은...
--------------------------------------------------------------------------------

[문서 4]
Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니

## 4. MMR (Maximal Marginal Relevance) 검색

MMR은 관련성과 다양성을 모두 고려하는 검색 방식입니다.

### 작동 원리

1. 먼저 `fetch_k`개의 후보 문서를 검색 (유사도 기준)
2. 후보 문서 중에서 이미 선택된 문서와 **다양성**을 고려하여 `k`개 선택
3. `lambda_mult` 파라미터로 관련성과 다양성의 균형 조절
   - `lambda_mult = 1.0`: 오직 관련성만 고려 (similarity search와 동일)
   - `lambda_mult = 0.0`: 오직 다양성만 고려
   - `lambda_mult = 0.5`: 관련성과 다양성 균형 (기본값)

> 🎯 **강의 포인트**: MMR은 RAG에서 매우 중요합니다. Similarity Search만 사용하면 비슷한 내용의 문서만 반환되어 LLM에 중복 정보가 제공될 수 있어요. MMR은 이 문제를 해결합니다.

> ⚠️ **자주 하는 실수**: `fetch_k`를 `k`보다 작게 설정하면 MMR이 정상 동작하지 않아요. `fetch_k`는 반드시 `k`보다 커야 합니다.

In [7]:
# MMR retriever 생성
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,  # 최종 반환 문서 수
        "fetch_k": 10,  # 후보 문서 수
        "lambda_mult": 0.6  # 관련성(0.6) vs 다양성(0.4)
    }
)

# 동일한 쿼리로 검색
mmr_docs = mmr_retriever.invoke(query)

print(f"📝 검색 쿼리: {query}")
print(f"\n🔍 MMR 검색 설정:")
print(f"  - 최종 반환 문서: {mmr_retriever.search_kwargs['k']}개")
print(f"  - 후보 문서: {mmr_retriever.search_kwargs['fetch_k']}개")
print(f"  - Lambda (관련성 가중치): {mmr_retriever.search_kwargs['lambda_mult']}")
print("\n" + "=" * 80)

for i, doc in enumerate(mmr_docs, 1):
    print(f"\n[MMR 문서 {i}]")
    print(doc.page_content[:200] + "...")
    print("-" * 80)


📝 검색 쿼리: Transformer 모델은 어떤 메커니즘을 기반으로 하나요?

🔍 MMR 검색 설정:
  - 최종 반환 문서: 3개
  - 후보 문서: 10개
  - Lambda (관련성 가중치): 0.6


[MMR 문서 1]
"Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다. 또한, 논문은 전통적인 RNN이나 CNN에 의존하지 않음으...
--------------------------------------------------------------------------------

[MMR 문서 2]
SARIMAX (계절성 자기회귀 누적 이동평균 외부 변수)

SARIMAX 모델은 SARIMA에 외부 설명 변수(또는 외생 변수)를 포함시킬 수 있는 확장된 형태입니다. 이 외부 변수들은 시계열 데이터에 영향을 미칠 수 있는 추가적인 정보(예: 경제 지표, 날씨 조건 등)를 모델에 제공합니다. SARIMAX 모델은 이러한 외부 변수들의 영향을 고려함으로써,...
--------------------------------------------------------------------------------

[MMR 문서 3]
Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이 포함되며, 다양한 언어에 대한 지원도 제공합니다. 라이브러리는 사전 훈련된 모델을 포함하고 있어, 사용자가 복잡한 모델을 처음부터 훈련시키지 않고도, 빠르게 고성능의 NLP 시스...
-------------------------------------------------------------------------

### MMR vs Similarity 비교

동일한 쿼리에 대해 두 검색 방식을 비교해봅시다.


In [8]:
# ---------------------------------------------------
# 동일 쿼리로 Similarity와 MMR 결과를 나란히 비교
# ---------------------------------------------------

# ============================================================
# TODO: 동일 쿼리에 sim_retriever와 mmr_retriever를 모두 실행하고 결과를 비교하세요
# 힌트: 두 retriever에 동일한 compare_query를 invoke()
# 예상 결과: 두 방식의 상위 문서 목록 — 중복 여부 차이 관찰
# ============================================================

# 비교 쿼리
compare_query = "NLP와 자연어 처리 기술에 대해 설명해주세요"

# 1. Similarity search
sim_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
sim_docs = sim_retriever.invoke(compare_query)

# 2. MMR search
mmr_retriever_compare = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5}
)
mmr_docs_compare = mmr_retriever_compare.invoke(compare_query)

print(f"📝 비교 쿼리: {compare_query}")
print("\n" + "=" * 80)
print("\n🔹 Similarity Search 결과 (관련성만 고려)")
print("=" * 80)
for i, doc in enumerate(sim_docs, 1):
    print(f"\n[문서 {i}] {doc.page_content[:150]}...")

print("\n" + "=" * 80)
print("\n🔹 MMR 결과 (관련성 + 다양성 고려)")
print("=" * 80)
for i, doc in enumerate(mmr_docs_compare, 1):
    print(f"\n[문서 {i}] {doc.page_content[:150]}...")

print("\n" + "=" * 80)
print("\n💡 차이점:")
print("  - Similarity: 가장 유사한 문서들만 반환 (중복될 수 있음)")
print("  - MMR: 유사하면서도 서로 다른 관점의 문서 반환")

📝 비교 쿼리: NLP와 자연어 처리 기술에 대해 설명해주세요


🔹 Similarity Search 결과 (관련성만 고려)

[문서 1] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사용될 수 있습니다. 이러한 언어의 특성으로 인해, NLP 시스템은 텍스트의 문법적 구조를 ...

[문서 2] NLP의 발전은 사람과 컴퓨터 간의 소통 방식을 근본적으로 변화시켰습니다. 예를 들어, 자연어를 이해할 수 있는 인터페이스를 통해 사용자는 복잡한 쿼리나 명령어를 사용하지 않고도 정보를 검색하거나 서비스를 이용할 수 있게 되었습니다. 또한, 자동 번역 시스템의 발전으로...

[문서 3] 기계 학습 분야에 있어서, Scikit-learn은 특히 초보자에게 친화적인 학습 자원을 제공함으로써, 복잡한 이론과 알고리즘을 이해하는 데 중요한 역할을 합니다. 이러한 접근성과 범용성은 Scikit-learn을 기계 학습을 시작하는 이들에게 인기 있는 선택지로 만들...


🔹 MMR 결과 (관련성 + 다양성 고려)

[문서 1] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사용될 수 있습니다. 이러한 언어의 특성으로 인해, NLP 시스템은 텍스트의 문법적 구조를 ...

[문서 2] HuggingFace

Hugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 소스 도구와 라이브러리를 제공합니다. 2016년에 설립된 이 회사는 특히 'Transfor...

[문서 3] 이 논문은 기계 학습 및 NLP 분야에서 기존의 접근 방식을 재고하고, 어텐션 메커니즘의 중요성을 강조함으로써 연구와 개발의 새로운 방향을 제시했습니다. 

## 5. Score Threshold 검색

유사도 점수가 특정 임계값 이상인 문서만 반환합니다.

### 언제 사용하면 좋을까요?

- 관련성이 낮은 문서를 필터링하고 싶을 때
- 정확도가 중요한 애플리케이션에서
- "관련 문서가 없으면 아무것도 반환하지 않음" 동작이 필요할 때

> 🔑 **핵심 개념**: Score Threshold는 "없는 것보다 나쁜 정보가 더 위험하다"는 원칙을 구현합니다. 임계값 미만이면 빈 결과를 반환해 LLM이 잘못된 정보를 바탕으로 답변하는 것을 방지해요.

> ⚠️ **자주 하는 실수**: `score_threshold`를 너무 높게 설정하면 정상적인 쿼리에서도 빈 결과가 반환됩니다. 0.6~0.8 범위에서 테스트하며 조정하세요.

In [9]:
# Score threshold retriever 생성
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.7  # 0.7 이상의 유사도를 가진 문서만 반환
    }
)

# 테스트 쿼리 1: 관련성 높은 쿼리
high_relevance_query = "Word2Vec은 무엇인가요?"
docs_high = threshold_retriever.invoke(high_relevance_query)

print(f"📝 쿼리 1 (관련성 높음): {high_relevance_query}")
print(f"검색된 문서 수: {len(docs_high)}개")
print("=" * 60)
for i, doc in enumerate(docs_high, 1):
    print(f"\n[문서 {i}]")
    print(doc.page_content[:200] + "...")

# 테스트 쿼리 2: 관련성 낮은 쿼리
low_relevance_query = "오늘 날씨는 어때요?"
docs_low = threshold_retriever.invoke(low_relevance_query)

print("\n" + "=" * 80)
print(f"\n📝 쿼리 2 (관련성 낮음): {low_relevance_query}")
print(f"검색된 문서 수: {len(docs_low)}개")
if len(docs_low) == 0:
    print("\n⚠️ 임계값 이상의 문서가 없습니다.")
else:
    for i, doc in enumerate(docs_low, 1):
        print(f"\n[문서 {i}]")
        print(doc.page_content[:200] + "...")

print("\n" + "=" * 80)
print("\n💡 Score Threshold의 장점:")
print("  - 낮은 관련성 문서 자동 필터링")
print("  - 검색 품질 보장")
print("  - 무의미한 결과 방지")


No relevant docs were retrieved using the relevance score threshold 0.7


📝 쿼리 1 (관련성 높음): Word2Vec은 무엇인가요?
검색된 문서 수: 0개


/Users/mhso/working/hankyung-llm-agent/02_langchain_basics/.venv/lib/python3.11/site-packages/langchain_core/vectorstores/base.py:1070: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='6b97f438-c91e-412a-9a08-c17e97f1557a', metadata={'source': './data/ai-story.txt'}, page_content='이 논문은 기계 학습 및 NLP 분야에서 기존의 접근 방식을 재고하고, 어텐션 메커니즘의 중요성을 강조함으로써 연구와 개발의 새로운 방향을 제시했습니다. "Attention Is All You Need"는 그 자체로 혁신적인 연구 성과일 뿐만 아니라, AI 기술의 발전에 있어 중대한 이정표로 평가받고 있습니다.\n\nARIMA (자기회귀 누적 이동평균), SARIMA (계절성 자기회귀 누적 이동평균), 그리고 SARIMAX (계절성 자기회귀 누적 이동평균 외부 변수)는 시계열 데이터 분석과 예측을 위한 통계 모델입니다. 이들은 시계열 데이터의 패턴을 이해하고 미래 값을 예측하는 데 널리 사용되며, 경제학, 금융, 기상학 등 다양한 분야에서 응용됩니다. 각 모델의 특징과 차이점을 교수처럼 전문적인 어조로 설명하겠습니다.\n\nARIMA (자기회귀 누적 이동평균)'), -0.2253526069651115), (Document(id='b64226f2-68ef-4840-b05d-a751989bebc8', metadata={'source': './data/ai-story.txt'}, page_content='Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과 모델의 공정성 및 투명성에 대한 중요성을 강조합니다. 회사는 AI 기술의 사회적 영향에 대해 적극적으로 논의하고, 모델 배포 시 발생할



📝 쿼리 2 (관련성 낮음): 오늘 날씨는 어때요?
검색된 문서 수: 0개

⚠️ 임계값 이상의 문서가 없습니다.


💡 Score Threshold의 장점:
  - 낮은 관련성 문서 자동 필터링
  - 검색 품질 보장
  - 무의미한 결과 방지


## 6. Top-k 설정

반환할 문서 개수를 동적으로 조절할 수 있습니다.


In [10]:
# ---------------------------------------------------
# k 값에 따른 검색 결과 수 변화 실험
# ---------------------------------------------------

# ============================================================
# TODO: k=1, 3, 5 각각으로 retriever를 생성하고 결과 수를 확인하세요
# 힌트: search_kwargs={"k": k} 로 k를 변수로 전달
# 예상 결과: k 값이 커질수록 반환 문서 수가 늘어남
# ============================================================

# 다양한 k 값으로 검색
test_query = "딥러닝과 머신러닝의 차이는 무엇인가요?"

# k=1, 3, 5 순서로 실험 — k가 클수록 더 많은 문서 검색
for k in [1, 3, 5]:
    retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
    docs_k = retriever_k.invoke(test_query)
    
    print(f"\n📊 k={k}일 때 검색 결과:")
    print("=" * 60)
    for i, doc in enumerate(docs_k, 1):
        print(f"[{i}] {doc.page_content[:100]}...")
    print()


📊 k=1일 때 검색 결과:
[1] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사...


📊 k=3일 때 검색 결과:
[1] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사...
[2] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...
[3] Word2Vec

Word2Vec은 자연어 처리(NLP) 분야에서 널리 사용되는 획기적인 단어 임베딩 기법 중 하나입니다. 2013년 Google의 연구팀에 의해 개발되었으며, 단...


📊 k=5일 때 검색 결과:
[1] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사...
[2] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...
[3] Word2Vec

Word2Vec은 자연어 처리(NLP) 분야에서 널리 사용되는 획기적인 단어 임베딩 기법 중 하나입니다. 2013년 Google의 연구팀에 의해 개발되었으며, 단...
[4] "Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 ...
[5] 이 논문은 기계 학습 및 NLP 분야에서 기존의 접근 방식을 재고하고, 어텐션 메커니즘의 중요성을 강조함으로써 연구와 개발의 새로운 방향을 제

## 7. 동적 설정 (ConfigurableField)

`ConfigurableField`를 사용하면 Retriever 생성 후에도 검색 파라미터를 동적으로 변경할 수 있습니다.

### 언제 사용하면 좋을까요?

- 사용자별로 다른 검색 설정을 적용할 때
- A/B 테스트로 최적의 검색 파라미터를 찾을 때
- 런타임에 검색 전략을 변경해야 할 때

> 💡 **실무 팁**: `ConfigurableField`는 프로덕션 환경에서 특히 유용합니다. 사용자 등급별로 k 값을 다르게 적용하거나, 특정 사용자에게만 MMR 검색을 제공하는 등 유연한 서비스가 가능해요.

In [11]:
from langchain_core.runnables import ConfigurableField

# 동적 설정이 가능한 retriever 생성
configurable_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
).configurable_fields(
    search_type=ConfigurableField(
        id="search_type",
        name="Search Type",
        description="검색 방식: similarity, mmr, similarity_score_threshold",
    ),
    search_kwargs=ConfigurableField(
        id="search_kwargs",
        name="Search Kwargs",
        description="검색 파라미터: k, fetch_k, lambda_mult, score_threshold 등",
    ),
)

print("✅ ConfigurableField를 가진 Retriever 생성 완료")


✅ ConfigurableField를 가진 Retriever 생성 완료


In [12]:
# ---------------------------------------------------
# ConfigurableField로 k=4 설정을 런타임에 적용
# ---------------------------------------------------

# ============================================================
# TODO: config={"configurable": {"search_kwargs": {"k": 4}}} 로 검색하세요
# 힌트: configurable_retriever.invoke(query, config=config1)
# 예상 결과: 4개 문서 반환
# ============================================================

# 예제 1: k=4로 검색
# configurable 딕셔너리 — 런타임에 설정을 덮어씀
config1 = {"configurable": {"search_kwargs": {"k": 4}}}

test_query_config = "HuggingFace는 무엇을 제공하나요?"
docs_config1 = configurable_retriever.invoke(test_query_config, config=config1)

print(f"📝 쿼리: {test_query_config}")
print(f"\n⚙️ 설정 1: k=4 (기본 similarity search)")
print(f"검색된 문서 수: {len(docs_config1)}개")
print("=" * 60)
for i, doc in enumerate(docs_config1, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

📝 쿼리: HuggingFace는 무엇을 제공하나요?

⚙️ 설정 1: k=4 (기본 similarity search)
검색된 문서 수: 4개
[1] Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과 모델의 공정성 및 투명성에 대한 중요성을 강조합니다. 회사는 AI 기술의 사회적 영향에 ...

[2] HuggingFace

Hugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 ...

[3] Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이...

[4] SciPy 라이브러리는 연구자와 엔지니어가 복잡한 수학적 문제를 효율적으로 해결할 수 있도록 설계되었습니다. 그 사용법은 상대적으로 간단하며, Python의 다른 과학 기술 스택과...



In [13]:
# ---------------------------------------------------
# 런타임에 검색 방식을 Score Threshold로 교체
# ---------------------------------------------------

# ============================================================
# TODO: search_type을 "similarity_score_threshold"로 바꾸고 threshold=0.75로 검색하세요
# 힌트: config2의 "search_type" 키에 새 방식을 문자열로 전달
# 예상 결과: 유사도 0.75 이상의 문서만 반환 (없으면 빈 리스트)
# ============================================================

# 예제 2: Score threshold 방식으로 변경
# search_type을 런타임에 교체 — 재생성 없이 검색 전략을 바꿀 수 있음
config2 = {
    "configurable": {
        "search_type": "similarity_score_threshold",
        "search_kwargs": {"score_threshold": 0.75}
    }
}

docs_config2 = configurable_retriever.invoke(test_query_config, config=config2)

print(f"\n⚙️ 설정 2: Score Threshold (threshold=0.75)")
print(f"검색된 문서 수: {len(docs_config2)}개")
print("=" * 60)
for i, doc in enumerate(docs_config2, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

No relevant docs were retrieved using the relevance score threshold 0.75



⚙️ 설정 2: Score Threshold (threshold=0.75)
검색된 문서 수: 0개


In [14]:
# 예제 3: MMR 방식으로 변경
config3 = {
    "configurable": {
        "search_type": "mmr",
        "search_kwargs": {
            "k": 3,
            "fetch_k": 10,
            "lambda_mult": 0.7
        }
    }
}

docs_config3 = configurable_retriever.invoke(test_query_config, config=config3)

print(f"\n⚙️ 설정 3: MMR (k=3, lambda=0.7)")
print(f"검색된 문서 수: {len(docs_config3)}개")
print("=" * 60)
for i, doc in enumerate(docs_config3, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("\n" + "=" * 80)
print("\n💡 ConfigurableField의 장점:")
print("  - 하나의 retriever로 여러 검색 전략 사용 가능")
print("  - 런타임에 파라미터 변경 가능")
print("  - A/B 테스트와 실험에 유용")



⚙️ 설정 3: MMR (k=3, lambda=0.7)
검색된 문서 수: 3개
[1] Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과 모델의 공정성 및 투명성에 대한 중요성을 강조합니다. 회사는 AI 기술의 사회적 영향에 ...

[2] HuggingFace

Hugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 ...

[3] Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이...



💡 ConfigurableField의 장점:
  - 하나의 retriever로 여러 검색 전략 사용 가능
  - 런타임에 파라미터 변경 가능
  - A/B 테스트와 실험에 유용


## 8. Query와 Document에 다른 임베딩 모델 사용하기

일부 임베딩 모델(예: Upstage Solar)은 쿼리용과 문서용 임베딩을 분리하여 제공합니다.

### 왜 분리할까요?

- **쿼리**: 짧고 검색 의도가 명확한 텍스트
- **문서**: 길고 정보가 풍부한 텍스트
- 각각에 최적화된 임베딩 모델을 사용하면 검색 품질이 향상됩니다.


In [15]:
# 주의: 이 예제를 실행하려면 UPSTAGE_API_KEY가 필요합니다.
# .env 파일에 UPSTAGE_API_KEY를 추가하세요.

upstage_vectorstore = None

if not UPSTAGE_API_KEY:
    print("⚠️ UPSTAGE_API_KEY가 없어 Upstage 임베딩 예제를 건너뜁니다.")
else:
    try:
        from langchain_upstage import UpstageEmbeddings
        
        # 1단계: 문서용 임베딩으로 벡터스토어 생성
        doc_embedder = UpstageEmbeddings(
            model="solar-embedding-1-large-passage",
            api_key=UPSTAGE_API_KEY,
        )
        
        # 문서 재로드 및 분할
        loader = TextLoader("./data/ai-story.txt")
        documents = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        split_docs = text_splitter.split_documents(documents)
        
        # 문서용 임베딩으로 벡터스토어 생성
        upstage_vectorstore = FAISS.from_documents(split_docs, doc_embedder)
        
        print("✅ Upstage 문서 임베딩으로 벡터스토어 생성 완료")
        print("모델: solar-embedding-1-large-passage (문서용)")
        
    except ImportError:
        print("⚠️ langchain-upstage 패키지가 설치되지 않았습니다.")
        print("설치 명령: pip install langchain-upstage")
    except Exception as e:
        print(f"⚠️ Upstage API 키가 설정되지 않았거나 오류가 발생했습니다: {e}")
        upstage_vectorstore = None


✅ Upstage 문서 임베딩으로 벡터스토어 생성 완료
모델: solar-embedding-1-large-passage (문서용)


In [16]:
if not UPSTAGE_API_KEY or upstage_vectorstore is None:
    print("⚠️ Upstage 예제를 실행할 수 없어 검색 단계를 건너뜁니다.")
else:
    try:
        # 2단계: 쿼리용 임베딩으로 검색
        query_embedder = UpstageEmbeddings(
            model="solar-embedding-1-large-query",
            api_key=UPSTAGE_API_KEY,
        )
        
        # 쿼리를 쿼리용 임베딩으로 변환
        query_text = "ARIMA 모델은 어떤 데이터 분석에 사용되나요?"
        query_vector = query_embedder.embed_query(query_text)
        
        # 벡터 유사도 검색
        docs = upstage_vectorstore.similarity_search_by_vector(query_vector, k=3)
        
        print(f"\n📝 쿼리: {query_text}")
        print("쿼리 임베딩 모델: solar-embedding-1-large-query")
        print(f"\n검색된 문서 수: {len(docs)}개")
        print("=" * 60)
        
        for i, doc in enumerate(docs, 1):
            print(f"\n[문서 {i}]")
            print(doc.page_content[:200] + "...")
        
        print("\n" + "=" * 80)
        print("\n💡 쿼리/문서 분리 임베딩의 장점:")
        print("  - 쿼리와 문서의 특성에 맞게 최적화")
        print("  - 검색 정확도 향상")
        print("  - 특히 짧은 쿼리와 긴 문서 매칭에 효과적")
        
    except Exception as exc:
        print(f"⚠️ Upstage 쿼리 임베딩 실행 중 오류: {exc}")



📝 쿼리: ARIMA 모델은 어떤 데이터 분석에 사용되나요?
쿼리 임베딩 모델: solar-embedding-1-large-query

검색된 문서 수: 3개

[문서 1]
ARIMA (자기회귀 누적 이동평균)

ARIMA 모델은 비계절적 시계열 데이터를 분석하고 예측하기 위해 설계되었습니다. 이 모델은 세 가지 주요 구성 요소를 기반으로 합니다: 자기회귀(AR) 부분, 차분(I) 부분, 그리고 이동평균(MA) 부분. AR 부분은 이전 관측값의 영향을 모델링하며, I 부분은 데이터를 안정화하는 데 필요한 차분의 횟수를 나타냅니...

[문서 2]
SARIMA (계절성 자기회귀 누적 이동평균)

SARIMA 모델은 ARIMA 모델을 확장한 것으로, 계절적 변동성을 포함한 시계열 데이터를 분석하고 예측하는 데 적합합니다. SARIMA는 추가적으로 계절성 자기회귀(SAR) 부분, 계절성 차분의 정도, 그리고 계절성 이동평균(SMA) 부분을 모델에 포함시킵니다. 이를 통해 모델은 비계절적 패턴뿐만 아니라 ...

[문서 3]
SARIMAX (계절성 자기회귀 누적 이동평균 외부 변수)

SARIMAX 모델은 SARIMA에 외부 설명 변수(또는 외생 변수)를 포함시킬 수 있는 확장된 형태입니다. 이 외부 변수들은 시계열 데이터에 영향을 미칠 수 있는 추가적인 정보(예: 경제 지표, 날씨 조건 등)를 모델에 제공합니다. SARIMAX 모델은 이러한 외부 변수들의 영향을 고려함으로써,...


💡 쿼리/문서 분리 임베딩의 장점:
  - 쿼리와 문서의 특성에 맞게 최적화
  - 검색 정확도 향상
  - 특히 짧은 쿼리와 긴 문서 매칭에 효과적


## 9. 검색 전략 비교 요약

### 검색 방식별 특징

| 검색 방식 | 장점 | 단점 | 추천 사용 시나리오 |
|---------|------|------|------------------|
| **Similarity** | 빠르고 간단, 높은 관련성 | 중복 결과 가능 | 일반적인 검색, 빠른 응답 필요 시 |
| **MMR** | 다양성 확보, 중복 감소 | 약간 느림 | 다양한 관점 필요, 탐색적 검색 |
| **Score Threshold** | 품질 보장, 무의미한 결과 방지 | 결과 없을 수 있음 | 정확도 중요, 낮은 관련성 필터링 |

### 파라미터 설정 가이드

- **k**: 일반적으로 3~5개가 적절 (너무 많으면 노이즈 증가)
- **fetch_k**: k의 2~4배 정도 (MMR에서)
- **lambda_mult**: 
  - 0.7~0.9: 관련성 우선 (정확도 중요 시)
  - 0.3~0.5: 다양성 우선 (탐색적 검색 시)
- **score_threshold**: 
  - 0.8~1.0: 매우 엄격 (고품질 필요 시)
  - 0.6~0.8: 적당한 필터링
  - 0.5 이하: 너무 느슨함

> 💡 **실무 팁**: 실제 서비스에서는 Similarity(속도)로 시작하고, 응답 품질 이슈가 생기면 MMR이나 Threshold를 추가하는 방식으로 단계적으로 개선하세요. 처음부터 복잡하게 만들 필요 없어요.

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| 검색 방식 | 언제 사용? | 핵심 파라미터 |
|----------|-----------|--------------|
| **Similarity** | 빠른 기본 검색 | `k` (반환 문서 수) |
| **MMR** | 다양성이 필요할 때 | `lambda_mult`, `fetch_k` |
| **Score Threshold** | 품질 기준 이상만 반환 | `score_threshold` (0~1) |
| **ConfigurableField** | 런타임 설정 변경 | `configurable` dict |

### 다음 노트북 예고

**02-ContextualCompressionRetriever**: 검색된 문서에서 관련 내용만 추출해 노이즈를 줄이는 방법을 배워요. 검색 품질과 LLM 비용을 동시에 개선하는 핵심 기법입니다.